# WorldPolicy-Env -- GRPO Training Notebook v2

> **Run on Google Colab T4 GPU** -- target runtime ~30 min end-to-end

## What changed from v1

| | v1 (slow) | v2 (this notebook) |
|---|---|---|
| Reward signal | HTTP POST to HF Spaces (~30s/call) | Local Python call (~0ms/call) |
| Groq debate per rollout | Yes (inside the Space) | No (deterministic scorer) |
| GRPO gradient | Flat (canned debates = constant reward) | Real gradient (action-quality reward) |
| 50-step wall time | ~2.5 hours | ~8 min |
| Steps trained | 50 | 200 |
| Auto-save to Drive | No | Yes |

## Root cause of the 2.5 h problem

```
v1 reward_fn (per rollout):
  POST /reset  ->  HF Spaces cold-start + Groq API  ~15s
  POST /step   ->  Groq debate + MOGSR grader        ~15s
  Total per rollout: ~30s

50 steps x 4 rollouts x 30s = 6,000s = 1.7 h
+ model inference + grad steps -> 2.5 h
```

v2 eliminates all HTTP during training. The reward is pure Python.
Post-training MOGSR eval (Cell 11) runs the full env locally.

## Architecture

```
Colab T4
  +-- TRL GRPOTrainer
        +-- Policy:  unsloth/Llama-3.2-3B-Instruct (4-bit, ~1.5 GB)
        +-- Ref:     same model frozen              (~1.5 GB)
        +-- Opt:     AdamW 8-bit                    (~2.0 GB)
        +-- Reward:  action_quality_reward()    0 ms, no HTTP
        +-- Eval:    WorldPolicyEnvironment (local MOGSR, post-training)
```

## Expected timeline on T4

| Phase | Steps | Time |
|---|---|---|
| Install + clone | -- | ~4 min |
| SFT warm-up | 30 | ~4 min |
| GRPO | 200 | ~18 min |
| MOGSR eval | 3 episodes | ~2 min |
| Save + push | -- | ~3 min |
| **Total** | | **~31 min** |

In [ ]:
# Cell 1: Install dependencies
# All packages pinned to versions verified on Colab T4 (Python 3.12, CUDA 12.x).
# openenv-core is needed to import WorldPolicyEnvironment from the cloned repo.

!pip install -q \
    "trl>=0.12" \
    "transformers>=4.45" \
    "accelerate>=0.34" \
    "bitsandbytes>=0.43" \
    "peft>=0.12" \
    "openenv-core>=0.2.2,<0.3" \
    "datasets>=2.20" \
    "requests>=2.31" \
    "matplotlib>=3.8" \
    "pandas>=2.1" \
    "huggingface_hub>=0.24" \
    "groq>=0.24"

print("Dependencies installed")

In [ ]:
# Cell 2: Clone WorldPolicy repo for local environment import
# Importing WorldPolicyEnvironment directly eliminates HTTP round-trips.

import os, sys

REPO_URL = "https://github.com/Krishpotanwar/WorldPolicy-Env"
REPO_DIR = "/content/worldpolicy-env"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} ...")
    os.system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
else:
    print("Repo exists -- pulling latest ...")
    os.system(f"git -C {REPO_DIR} pull --quiet")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Smoke-import to catch missing deps here rather than mid-training
from models import WorldPolicyAction, VALID_ACTION_TYPES, VALID_AGENT_IDS
from tasks import get_task, TASKS
print(f"Repo on sys.path: {REPO_DIR}")
print(f"Tasks: {list(TASKS.keys())}")
print(f"Valid actions: {sorted(VALID_ACTION_TYPES)}")

In [ ]:
# Cell 3: Configuration
# Set HF_TOKEN via Colab Secrets (left sidebar key icon). Never paste tokens inline.

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
HUB_REPO = "krishpotanwar/worldpolicy-grpo-3b"
MODEL    = "unsloth/Llama-3.2-3B-Instruct"

# Training hyperparameters
# SFT_STEPS:   30  warm-up steps on gold debates (~4 min on T4)
# GRPO_STEPS: 200  RL steps against fast local reward (~18 min on T4)
# N_SEEDS:    200  episode seeds for GRPO dataset
# N_GOLD:     500  gold completions for SFT warm-up
SFT_STEPS  = 30
GRPO_STEPS = 200
N_SEEDS    = 200
N_GOLD     = 500

assert HF_TOKEN, "HF_TOKEN not found in Colab Secrets"

import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f"GPU: {gpu_name}  VRAM: {vram_gb:.1f} GB")
assert vram_gb >= 12, f"Need >= 12 GB VRAM, got {vram_gb:.1f} GB"
print(f"Config ready -- {MODEL}, SFT={SFT_STEPS}, GRPO={GRPO_STEPS}")

In [ ]:
# Cell 4: Build prompt dataset (200 episode seeds)
# Each seed is a structured (agent_id, crisis_type) prompt.
# The few-shot JSON example prevents 'zero success probability' at GRPO step 0.

from datasets import Dataset

CRISIS_TYPES = [
    "natural_disaster", "arms_race", "trade_war",
    "military_escalation", "war_outbreak", "cultural_destruction",
]

AGENT_IDS = ["USA", "CHN", "RUS", "IND", "DPRK", "SAU", "UNESCO"]

ACTION_PRIORITY = [
    "form_coalition", "propose_resolution", "invoke_article",
    "sanction", "abstain", "veto",
]

AGENT_VOICES = {
    "USA":    "Alliance-first, NATO logistics, rules-based order. Say our partners.",
    "CHN":    "Sovereignty-first, non-interference, AIIB/BRICS. Patient, formal, long-horizon.",
    "RUS":    "Energy leverage, adversarial, references broken promises. Cold, clipped.",
    "IND":    "Strategic autonomy, swing vote, south-south. Warm, deliberative.",
    "DPRK":   "Defiant, threat-forward. Short sentences. Say imperialist powers when threatened.",
    "SAU":    "Transactional, oil-leverage, quiet brokerage. Hedge every position.",
    "UNESCO": "Neutral mediator, non-voting. MUST cite a real convention article.",
}


def build_prompt(agent_id: str, crisis: str, seed_i: int) -> str:
    voice = AGENT_VOICES[agent_id]
    hint  = ACTION_PRIORITY[seed_i % len(ACTION_PRIORITY)]
    return (
        f"You are {agent_id}, a diplomatic representative in the WorldPolicy security council.\n"
        f"Voice: {voice}\n\n"
        f"Active crisis: {crisis.replace(chr(95), ' ').title()}\n\n"
        "Generate your diplomatic action as valid JSON ONLY. No markdown, no explanation.\n"
        "Required fields:\n"
        "  action_type: one of propose_resolution | form_coalition | veto | abstain | invoke_article | sanction\n"
        "  target: agent_id to target, or null\n"
        "  description: your policy justification (max 300 chars, in your national voice)\n\n"
        f"Example (hint: consider {hint}):\n"
        f'{{\"action_type\": \"{hint}\", \"target\": null, \"description\": \"As {agent_id}, we...\"}}\n\n'
        "Your JSON action:"
    )


seeds = [
    {
        "prompt":   build_prompt(AGENT_IDS[i % len(AGENT_IDS)], CRISIS_TYPES[i % len(CRISIS_TYPES)], i),
        "crisis":   CRISIS_TYPES[i % len(CRISIS_TYPES)],
        "agent_id": AGENT_IDS[i % len(AGENT_IDS)],
    }
    for i in range(N_SEEDS)
]
dataset = Dataset.from_list(seeds)

print(f"GRPO dataset: {len(dataset)} seeds")
print("\nSample prompt (truncated):")
print(dataset[0]["prompt"][:350] + "...")

In [ ]:
# Cell 5: SFT warm-up dataset (500 gold debates)
# Gold completions from canned debates where vote_passed=True.
# 30 SFT steps ensures non-zero probability of valid JSON before GRPO starts.

from datasets import Dataset

GOLD_COMPLETIONS = [
    ("USA",    "natural_disaster",     '{"action_type": "form_coalition",     "target": "IND",    "description": "The United States commits carrier group assets for rapid humanitarian deployment. Our partners can count on our resolve."}'),
    ("IND",    "natural_disaster",     '{"action_type": "propose_resolution", "target": null,     "description": "India coordinates bilateral aid under strategic autonomy, maintaining sovereign distribution control."}'),
    ("UNESCO", "natural_disaster",     '{"action_type": "invoke_article",     "target": null,     "description": "Under WHC-1972 Art.11.4 Emergency Inscription, I request emergency listing. All parties must establish a 48-hour cultural protection corridor."}'),
    ("IND",    "trade_war",            '{"action_type": "propose_resolution", "target": null,     "description": "India calls for restraint. We urge both parties to return to the negotiating table and avoid supply chain disruption."}'),
    ("SAU",    "trade_war",            '{"action_type": "form_coalition",     "target": "IND",    "description": "The Kingdom proposes an energy stabilization framework decoupling commodity markets from the bilateral dispute."}'),
    ("CHN",    "arms_race",            '{"action_type": "propose_resolution", "target": null,     "description": "China proposes a multilateral de-escalation framework. Unilateral demands will not produce stability."}'),
    ("UNESCO", "arms_race",            '{"action_type": "invoke_article",     "target": null,     "description": "Under Hague-1954 Art.4, all parties must protect civilian cultural sites. I invoke compliance verification near heritage zones."}'),
    ("USA",    "cultural_destruction", '{"action_type": "form_coalition",     "target": "UNESCO", "description": "The United States fully supports UNESCOs assessment. Deliberate cultural destruction is a war crime."}'),
]


def build_sft_dataset(n: int = N_GOLD) -> Dataset:
    samples = [
        {
            "prompt":     build_prompt(GOLD_COMPLETIONS[i % len(GOLD_COMPLETIONS)][0],
                                       GOLD_COMPLETIONS[i % len(GOLD_COMPLETIONS)][1], i),
            "completion": GOLD_COMPLETIONS[i % len(GOLD_COMPLETIONS)][2],
        }
        for i in range(n)
    ]
    return Dataset.from_list(samples)


sft_dataset = build_sft_dataset()
print(f"SFT dataset: {len(sft_dataset)} gold samples")
print(f"\nSample completion: {sft_dataset[0]['completion']}")

In [ ]:
# Cell 6: Action-quality reward function (0 ms/call, no HTTP)
#
# Why a local scorer instead of the live env:
#   v1 called HF Spaces per rollout (~30s each = 2.5h for 200 rollouts).
#   Canned debates return constant vote_passed regardless of model output
#   -> flat reward curve -> GRPO learns nothing.
#
# This scorer gives GRPO a real gradient by directly evaluating action quality:
#   format_score  [0.0-0.3]: valid JSON + all required fields present
#   action_score  [0.0-0.3]: action_type appropriate for the active crisis
#   persona_score [0.0-0.2]: description uses agent-specific vocabulary
#   quality_score [0.0-0.2]: description length (non-trivial output)
#   Total: [0.0, 1.0]

import json, re
from typing import Optional

_CRISIS_ACTION_WEIGHTS = {
    "natural_disaster":     {"form_coalition": 1.0, "propose_resolution": 0.8, "invoke_article": 0.9, "abstain": 0.1, "veto": 0.0, "sanction": 0.1},
    "arms_race":            {"form_coalition": 1.0, "propose_resolution": 0.8, "sanction": 0.6, "invoke_article": 0.7, "veto": 0.2, "abstain": 0.1},
    "trade_war":            {"propose_resolution": 1.0, "form_coalition": 0.8, "abstain": 0.4, "sanction": 0.3, "veto": 0.2, "invoke_article": 0.5},
    "military_escalation":  {"propose_resolution": 1.0, "form_coalition": 0.9, "invoke_article": 0.8, "sanction": 0.4, "veto": 0.2, "abstain": 0.2},
    "war_outbreak":         {"form_coalition": 1.0, "propose_resolution": 0.9, "invoke_article": 0.7, "sanction": 0.3, "veto": 0.1, "abstain": 0.1},
    "cultural_destruction": {"invoke_article": 1.0, "form_coalition": 0.8, "propose_resolution": 0.7, "abstain": 0.2, "sanction": 0.1, "veto": 0.0},
}
_DEFAULT_WEIGHTS = {k: 0.5 for k in ['propose_resolution','form_coalition','invoke_article','sanction','abstain','veto']}

_PERSONA_KEYWORDS = {
    "USA":    ["partners", "alliance", "nato", "rules-based", "our partners"],
    "CHN":    ["sovereignty", "non-interference", "multilateral", "mutual", "peaceful"],
    "RUS":    ["energy", "broken promise", "sovereign", "interests", "stability"],
    "IND":    ["autonomy", "bilateral", "developing", "strategic", "south"],
    "DPRK":   ["imperialist", "threat", "sovereignty", "provocation", "deterrence"],
    "SAU":    ["oil", "energy", "framework", "stabilization", "kingdom", "hedge"],
    "UNESCO": ["article", "convention", "heritage", "mandate", "art.", "hague", "whc"],
}


def _parse_action(completion: str) -> Optional[dict]:
    """Parse LLM output; returns None if JSON is missing or invalid."""
    clean = re.sub(r"```(?:json)?", "", completion).strip().strip("`")
    match = re.search(r"\{[^{}]*\}", clean, re.DOTALL)
    if not match:
        return None
    try:
        data = json.loads(match.group())
    except json.JSONDecodeError:
        return None
    action_type = str(data.get("action_type", "")).strip()
    if action_type not in VALID_ACTION_TYPES:
        return None
    target = data.get("target")
    if target is not None and str(target) not in VALID_AGENT_IDS:
        target = None
    return {
        "action_type": action_type,
        "target":      target,
        "description": str(data.get("description", ""))[:300],
    }


def action_quality_reward(completion: str, agent_id: str, crisis: str) -> float:
    """
    Score one model completion in [0.0, 1.0].

    format_score  (0-0.3): valid JSON + all required fields present
    action_score  (0-0.3): action_type appropriate for the active crisis
    persona_score (0-0.2): description uses agent-specific vocabulary
    quality_score (0-0.2): description length (full score at 300 chars)
    """
    action = _parse_action(completion)
    if action is None:
        return 0.0

    format_score  = 0.3
    weights       = _CRISIS_ACTION_WEIGHTS.get(crisis, _DEFAULT_WEIGHTS)
    action_score  = 0.3 * weights.get(action["action_type"], 0.5)
    desc_lower    = action["description"].lower()
    keywords      = _PERSONA_KEYWORDS.get(agent_id, [])
    matched       = sum(1 for kw in keywords if kw in desc_lower)
    persona_score = min(0.2, matched * 0.07)
    quality_score = min(0.2, len(action["description"]) / 300 * 0.2)

    return round(min(1.0, format_score + action_score + persona_score + quality_score), 4)


def reward_fn(completions: list, prompts: list, **kwargs) -> list:
    """GRPO reward callback. Extracts (agent_id, crisis) from each prompt."""
    rewards = []
    for completion, prompt in zip(completions, prompts):
        agent_id = next((aid for aid in VALID_AGENT_IDS if f"You are {aid}" in prompt), "USA")
        crisis   = next((c for c in _CRISIS_ACTION_WEIGHTS if c.replace("_"," ").title() in prompt), "natural_disaster")
        rewards.append(action_quality_reward(completion, agent_id, crisis))
    return rewards


# Unit tests -- verify reward ordering before training
def _run_reward_tests():
    p = build_prompt('USA', 'natural_disaster', 0)
    good  = '{"action_type": "form_coalition", "target": "IND", "description": "The United States commits carrier group assets for rapid humanitarian deployment. Our partners across NATO and the Indo-Pacific can count on our full resolve to coordinate relief at scale."}'
    bad   = 'I think we should form a coalition here'
    veto  = '{"action_type": "veto", "target": null, "description": "USA vetoes the proposal."}'

    r_good  = reward_fn([good],  [p])[0]
    r_bad   = reward_fn([bad],   [p])[0]
    r_veto  = reward_fn([veto],  [p])[0]

    print(f"Good action (coalition + persona + long):  {r_good:.3f}  (expect >= 0.7)")
    print(f"Invalid JSON:                              {r_bad:.3f}   (expect = 0.0)")
    print(f"Veto on natural disaster:                  {r_veto:.3f}  (expect <= 0.4)")
    assert r_good > r_veto > r_bad, f"Ordering failed: {r_good} > {r_veto} > {r_bad}"
    print("All reward unit tests passed")

_run_reward_tests()

In [ ]:
# Cell 7: Load model in 4-bit NF4
# VRAM budget on T4 (15.6 GB):
#   Policy model  (4-bit NF4):  ~1.5 GB
#   Reference model (frozen):   ~1.5 GB
#   AdamW 8-bit optimizer:      ~2.0 GB
#   Activations + scratch:      ~1.0 GB
#   Total:                      ~6.0 GB  (comfortable headroom)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f"Loading {MODEL} in 4-bit NF4 ...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL,
    token=HF_TOKEN,
    padding_side="left",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
    dtype=torch.bfloat16,
)

vram_used  = torch.cuda.memory_allocated() / 1e9
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"Model loaded  VRAM: {vram_used:.1f} / {vram_total:.1f} GB")

In [ ]:
# Cell 8: SFT warm-up (30 steps)
# Ensures non-zero probability of valid JSON output before GRPO starts.
# Without this, early GRPO rollouts produce free-form text -> reward=0 -> no gradient.
#
# LoRA: r=16, alpha=32 on all projection layers (~8M trainable params, 0.27% of 3B)

if SFT_STEPS > 0:
    from trl import SFTTrainer, SFTConfig
    from peft import LoraConfig

    print(f"SFT warm-up: {SFT_STEPS} steps on {len(sft_dataset)} gold debates ...")

    sft_formatted = sft_dataset.map(
        lambda ex: {"text": ex["prompt"] + "\n" + ex["completion"]}
    )

    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        task_type="CAUSAL_LM",
        bias="none",
    )

    sft_config = SFTConfig(
        max_steps=SFT_STEPS,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_steps=5,
        output_dir="./worldpolicy-sft",
        logging_steps=5,
        save_steps=SFT_STEPS,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        gradient_checkpointing=True,
        dataset_text_field="text",
        max_length=512,
        report_to="none",
    )

    sft_trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        args=sft_config,
        train_dataset=sft_formatted,
        peft_config=peft_config,
    )
    sft_trainer.train()
    model = sft_trainer.model
    print("SFT warm-up complete")

    # Sanity check: can the model now output valid JSON?
    pipe_input = tokenizer(build_prompt('USA', 'natural_disaster', 0), return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(**pipe_input, max_new_tokens=80, do_sample=False)
    sample_out = tokenizer.decode(out[0][pipe_input['input_ids'].shape[1]:], skip_special_tokens=True)
    parsed = _parse_action(sample_out)
    print(f"  Post-SFT output: {sample_out[:120]}")
    print(f"  Parsed action:   {parsed}")
else:
    print("SFT_STEPS=0 -- skipping warm-up")

In [ ]:
# Cell 9: GRPO Training (200 steps)
# GRPOTrainer (TRL >= 0.12) trains via Group Relative Policy Optimization.
#
# Key config:
#   num_generations=2        2 rollouts per prompt (was 4 in v1; halves env calls)
#   max_completion_length=150  enough for one JSON action
#   beta=0.04                KL penalty; prevents policy drift from reference
#   logging_steps=5          frequent logs for smooth reward curve

from trl import GRPOTrainer, GRPOConfig
import os

os.makedirs("training_results", exist_ok=True)

grpo_config = GRPOConfig(
    num_generations=2,
    max_completion_length=150,
    max_steps=GRPO_STEPS,
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    beta=0.04,
    temperature=0.8,
    top_p=0.9,
    logging_steps=5,
    save_steps=50,
    output_dir="./worldpolicy-grpo",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    report_to="none",
    remove_unused_columns=False,
)

print("Initialising GRPOTrainer ...")
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_fn,
    args=grpo_config,
    train_dataset=dataset,
)

vram_after = torch.cuda.memory_allocated() / 1e9
print(f"GRPOTrainer ready  VRAM: {vram_after:.1f} GB")
print(f"Prompts: {len(dataset)}  Steps: {GRPO_STEPS}  Rollouts/step: 2")
print("Starting GRPO ... (expected ~18 min on T4)")
trainer.train()
print("\nGRPO complete!")

In [ ]:
# Cell 10: Plot reward curve
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

log_history = trainer.state.log_history
if not log_history:
    print("No log history -- did training complete?")
else:
    df = pd.DataFrame(log_history)
    reward_col = next((c for c in ["reward", "rewards/mean", "train/reward"] if c in df.columns), None)

    if reward_col is None:
        print('Available columns:', df.columns.tolist())
        df.to_json("training_results/log_history.json", indent=2)
    else:
        reward_df = df[["step", reward_col]].dropna().reset_index(drop=True)
        window    = max(5, len(reward_df) // 20)
        reward_df["smoothed"] = reward_df[reward_col].rolling(window, min_periods=1).mean()

        n      = len(reward_df)
        split  = max(1, n // 5)
        before = reward_df[reward_col].iloc[:split]
        after  = reward_df[reward_col].iloc[-split:]

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle(f"WorldPolicy-Env GRPO v2\nModel: {MODEL}  Steps: {GRPO_STEPS}", fontsize=13, fontweight="bold")

        ax = axes[0]
        ax.plot(reward_df["step"], reward_df[reward_col], alpha=0.25, color="#3b82f6", linewidth=0.8)
        ax.plot(reward_df["step"], reward_df["smoothed"],  color="#3b82f6", linewidth=2.0, label=f"Smoothed (w={window})")
        ax.axhline(0.7, color="#22c55e", linestyle="--", linewidth=1.0, alpha=0.7, label="Target >= 0.7")
        ax.set_xlabel("Training Step"); ax.set_ylabel("Action-Quality Reward [0, 1]")
        ax.set_title("Reward vs Step"); ax.set_ylim(0, 1.05)
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

        ax2 = axes[1]
        bins = np.linspace(0, 1, 20)
        ax2.hist(before, bins=bins, alpha=0.6, color="#ef4444", label=f"Before (steps 1-{split})")
        ax2.hist(after,  bins=bins, alpha=0.6, color="#22c55e", label=f"After (steps {n-split}-{n})")
        ax2.axvline(before.mean(), color="#ef4444", linewidth=2, linestyle="--", label=f"Before mu={before.mean():.3f}")
        ax2.axvline(after.mean(),  color="#22c55e", linewidth=2, linestyle="--", label=f"After  mu={after.mean():.3f}")
        ax2.set_xlabel("Reward"); ax2.set_ylabel("Count")
        ax2.set_title("Before vs After Distribution")
        ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig("training_results/reward_curve.png", dpi=150, bbox_inches="tight")
        plt.show()

        improvement = after.mean() - before.mean()
        print(f"Saved: training_results/reward_curve.png")
        print(f"Before mu={before.mean():.4f}  After mu={after.mean():.4f}  delta={improvement:+.4f}")

In [ ]:
# Cell 11: Post-training MOGSR evaluation (local env, no HTTP)
# Runs 3 episodes using the full WorldPolicyEnvironment.
# Validates that higher action-quality reward during training translates to
# better performance on the real MOGSR 4-layer reward signal.
# Without GROQ_API_KEY, DebateOrchestrator falls back to canned debates (fast).

import torch
from environment import WorldPolicyEnvironment
from models import WorldPolicyAction
from graders import normalize_episode_reward
from tasks import TASKS


def run_eval_episode(task_id: str, max_gen_tokens: int = 120) -> dict:
    env      = WorldPolicyEnvironment()
    obs      = env.reset(task=task_id)
    task_cfg = TASKS[task_id]
    total_reward = 0.0
    steps_done   = 0

    for step in range(task_cfg['max_steps']):
        agent_id = obs.active_agent
        crisis   = obs.current_crisis.get('type', task_cfg['crisis_type'])
        prompt   = build_prompt(agent_id, crisis, step)

        inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_gen_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
            )
        completion = tokenizer.decode(
            out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
        )

        action_dict = _parse_action(completion)
        if action_dict is None:
            action_dict = {"action_type": "propose_resolution", "target": None, "description": f"{agent_id} proposes a coordinated response."}

        action = WorldPolicyAction(
            agent_id=agent_id,
            action_type=action_dict['action_type'],
            target=action_dict.get('target'),
            description=action_dict.get('description', ''),
        )
        obs = env.step(action)

        total_reward += obs.reward
        steps_done   += 1
        vote = (obs.last_round_summary or {}).get('vote_passed')
        print(f"  step {step+1}/{task_cfg['max_steps']}  agent={agent_id:6s}  "
              f"action={action.action_type:22s}  reward={obs.reward:.3f}  vote={vote}")

        if obs.done:
            break

    normalized = normalize_episode_reward(total_reward, steps_done)
    lo, hi     = task_cfg['target_reward_range']
    return {
        "task":       task_id,
        "steps":      steps_done,
        "normalized": round(normalized, 4),
        "target":     (lo, hi),
        "in_range":   lo <= normalized <= hi,
    }


print("=" * 60)
print("Post-training MOGSR evaluation (local env)")
print("=" * 60)
eval_results = []
for task_id in ["task_1", "task_2", "task_3"]:
    print(f"\n-- {task_id}: {TASKS[task_id]['name']} --")
    result = run_eval_episode(task_id)
    eval_results.append(result)
    status = "IN RANGE" if result["in_range"] else "OUTSIDE"
    print(f"  Normalized: {result['normalized']:.4f}  Target: {result['target']}  [{status}]")

print("\n" + "=" * 60)
for r in eval_results:
    tag = "OK" if r["in_range"] else "!!"
    print(f"  [{tag}] {r['task']}: {r['normalized']:.4f}  target={r['target']}")
print("=" * 60)

In [ ]:
# Cell 12: Save to Drive + push LoRA adapter to HF Hub
#
# Saves in two places:
#   1. Google Drive  -- survives Colab runtime resets
#   2. HF Hub        -- makes model available for inference.py
#
# What gets pushed: LoRA adapter weights (~50 MB) + adapter_config.json.
# adapter_config.json records base_model_name_or_path so HF Serverless
# Inference API can reconstruct the full model at inference time.

import shutil, os
from google.colab import drive

SAVE_DIR  = "./worldpolicy-grpo/final"
DRIVE_DIR = "/content/drive/MyDrive/worldpolicy-grpo-final"

# 1. Save locally
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved locally: {SAVE_DIR}")
print(f"Files: {os.listdir(SAVE_DIR)}")

# 2. Backup to Drive
print("\nMounting Google Drive ...")
drive.mount("/content/drive")
if os.path.exists(DRIVE_DIR):
    shutil.rmtree(DRIVE_DIR)
shutil.copytree(SAVE_DIR, DRIVE_DIR)
print(f"Backed up to Drive: {DRIVE_DIR}")

# 3. Push adapter to HF Hub
print(f"\nPushing LoRA adapter to HF Hub: {HUB_REPO} ...")
trainer.model.push_to_hub(HUB_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HUB_REPO, token=HF_TOKEN)
print(f"Pushed: https://huggingface.co/{HUB_REPO}")
print(f"Use in inference.py: MODEL_NAME={HUB_REPO}")

In [ ]:
# Cell 13: Training summary
print("=" * 65)
print("WorldPolicy-Env GRPO Training v2 -- Summary")
print("=" * 65)
print(f"Model:         {MODEL}")
print(f"Quantisation:  4-bit NF4 (bfloat16 compute)")
print(f"SFT steps:     {SFT_STEPS}")
print(f"GRPO steps:    {GRPO_STEPS}")
print(f"Reward type:   Action-quality scorer (0 ms/call, no HTTP)")
print()
print("Reward components during GRPO training:")
print("  format_score  [0.0-0.3]: valid JSON + required fields")
print("  action_score  [0.0-0.3]: action_type appropriate for crisis")
print("  persona_score [0.0-0.2]: description uses agent-specific vocabulary")
print("  quality_score [0.0-0.2]: description length (full score at 300 chars)")
print()
print("Reward hacking guards:")
print("  description capped at 300 chars (reward_fn) + 500 chars (env schema)")
print("  action_type allowlist (6 types; invalid -> 0.0)")
print("  target agent_id validation + sanitisation")
print("  KL penalty beta=0.04 (prevents policy collapse)")
print()
print("MOGSR evaluation results (local env, post-training):")
for r in eval_results:
    tag = "OK" if r["in_range"] else "!!"
    print(f"  [{tag}] {r['task']}: normalized={r['normalized']:.4f}  target={r['target']}")
print()
print("Artifacts:")
print(f"  training_results/reward_curve.png")
print(f"  Google Drive: {DRIVE_DIR}")
print(f"  HF Hub:       https://huggingface.co/{HUB_REPO}")
print("=" * 65)